# Coverless steganography baseline

Before running: turn on **GPU T4 x2** and **Internet**, and add the dataset. See `MANUAL.md`, section 4.

Change only the settings in the next cell.

In [ ]:
# ===== SETTINGS: change only these lines =====
REPO = "i-git-abhishek/Enhanced-Coverless-Steganography"
BRANCH = "rewrite-baseline"
CONFIG = "configs/voc2007.yaml"
RUN_EXTRACTION = True         # False if a finished cache dataset is attached
RUN_BASELINE = True           # one run with the n and M in CONFIG
SWEEP_N = []                  # for example [3, 4, 5, 6]
SWEEP_M = []                  # for example [3, 4, 5, 6]
RUN_ORDERING = False          # Phase 2 ordering experiment on CONFIG (CPU, needs the cache)
ORDERING_N = [3, 4, 5, 6]     # n values for the ordering experiment
ORDERING_M = [3, 5, 6]        # extra M values (n from CONFIG)
RUN_PHASE3 = False            # Phase 3: SSCD descriptors, new retrieval, retrievability check (GPU, needs the cache)
PHASE3_MODES = ["sscd", "sscd_align"]   # each one is a full baseline run without the cluster ID
USABLE_THRESHOLDS = [0.9]     # sscd_align runs that only send images passing the check; [] to skip the check


In [ ]:
import os, subprocess

def run(command):
    print("\n$ " + command, flush=True)
    subprocess.run(command, shell=True, check=True)

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # no secret is needed for a public repo

folder = "/kaggle/working/" + REPO.split("/")[1]
if not os.path.isdir(folder):
    auth = f"x-access-token:{token}@" if token else ""
    # check=True is not used here so that a failed command cannot print the token.
    result = subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, f"https://{auth}github.com/{REPO}.git", folder], capture_output=True)
    if result.returncode != 0:
        raise SystemExit("Clone failed. Check REPO and BRANCH. For a private repo, add the GITHUB_TOKEN secret (MANUAL.md section 4).")
    subprocess.run(["git", "-C", folder, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir(folder)
run("git log -1 --oneline")
subprocess.run("nvidia-smi", shell=True)  # absent when the GPU is off


In [ ]:
# Stop early if the code or data has a problem.
run("python -u run.py smoke")
run(f"python -u run.py check-data --config {CONFIG}")


In [ ]:
# Keep every finished cache from the inputs in this run's output, so later versions can reuse it.
import glob, shutil
for done in glob.glob("/kaggle/input/**/cache/*/COMPLETE", recursive=True):
    source = os.path.dirname(done)
    target = os.path.join("cache", os.path.basename(source))
    if not os.path.exists(target):
        shutil.copytree(source, target)
        print("Kept cache:", target)
for weights in glob.glob("/kaggle/input/**/sscd_disc_mixup.torchscript.pt", recursive=True):
    os.makedirs("cache/weights", exist_ok=True)
    if not os.path.exists("cache/weights/sscd_disc_mixup.torchscript.pt"):
        shutil.copy(weights, "cache/weights/")
        print("Kept SSCD weights")
print("Caches available:", sorted(os.listdir("cache")) if os.path.isdir("cache") else "none")


In [ ]:
# Detector and ResNet-101 pass. A finished cache in an attached dataset is reused automatically.
if RUN_EXTRACTION:
    run(f"python -u run.py extract --config {CONFIG} --limit 4")
    run(f"python -u run.py extract --config {CONFIG}")


In [ ]:
# The stored detections must match a fresh detector pass. A mismatch does not stop the notebook,
# but must be read and reported (MANUAL.md section 11).
if RUN_EXTRACTION:
    subprocess.run(f"python -u run.py verify-detections --config {CONFIG} --sample 100", shell=True)


In [ ]:
# Phase 3 GPU passes. Both are cached and skipped when a finished copy is attached.
# The SSCD weights are downloaded once (needs Internet), or taken from an attached dataset.
if RUN_PHASE3:
    run(f"python -u run.py descriptors --config {CONFIG}")
    if USABLE_THRESHOLDS:
        run(f"python -u run.py retrievability --config {CONFIG} --set retrieval=sscd_align")

In [ ]:
# A failed setting must not lose finished runs: the next cell still creates the bundle.
failures = []

def attempt(command):
    print("\n$ " + command, flush=True)
    if subprocess.run(command, shell=True).returncode != 0:
        failures.append(command)

if RUN_BASELINE:
    attempt(f"python -u run.py baseline --config {CONFIG}")
if SWEEP_N:
    attempt(f"python -u run.py sweep --config {CONFIG} --param n --values " + " ".join(map(str, SWEEP_N)))
if SWEEP_M:
    attempt(f"python -u run.py sweep --config {CONFIG} --param M --values " + " ".join(map(str, SWEEP_M)))
if RUN_ORDERING:
    for n in ORDERING_N:
        attempt(f"python -u run.py ordering --config {CONFIG} --set n={n}")
    for m in ORDERING_M:
        attempt(f"python -u run.py ordering --config {CONFIG} --set M={m}")
if RUN_PHASE3:
    for mode in PHASE3_MODES:
        attempt(f"python -u run.py baseline --config {CONFIG} --set retrieval={mode}")
    for threshold in USABLE_THRESHOLDS:
        attempt(f"python -u run.py baseline --config {CONFIG} --set retrieval=sscd_align --set usable_threshold={threshold}")

In [ ]:
# Creates the one file to download from the Output tab.
run("python -u run.py bundle --output /kaggle/working/results_bundle.tar.gz")
for name in ["experiments/RESULTS_DASHBOARD.md", "experiments/ORDERING_DASHBOARD.md"]:
    if os.path.exists(name):
        print("\n===== " + name + " =====\n")
        print(open(name, encoding="utf-8").read()[:6000])
if failures:
    raise SystemExit("These commands failed. Finished runs are still in the bundle:\n" + "\n".join(failures))
